In [127]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [128]:
df=pd.read_csv("qoute_dataset.csv")

In [129]:
quotes=df["quote"]
quotes.head()


0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: str

In [130]:
quotes=quotes.str.lower()
quotes.head()


0    “the world as we have created it is a process ...
1    “it is our choices, harry, that show what we t...
2    “there are only two ways to live your life. on...
3    “the person, be it gentleman or lady, who has ...
4    “imperfection is beauty, madness is genius and...
Name: quote, dtype: str

In [131]:
import string
translator=str.maketrans("","",string.punctuation)
quotes=quotes.apply(lambda x: x.translate(translator))

In [132]:
quotes.head()


0    “the world as we have created it is a process ...
1    “it is our choices harry that show what we tru...
2    “there are only two ways to live your life one...
3    “the person be it gentleman or lady who has no...
4    “imperfection is beauty madness is genius and ...
Name: quote, dtype: str

In [133]:
from tensorflow.keras.preprocessing.text import Tokenizer


In [134]:
vocab_size=8980
tokenizer=Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)


In [135]:
word_index=tokenizer.word_index
print("Number of unique words:",len(word_index))
list(word_index.items())[:10]

Number of unique words: 8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [136]:
sequences=tokenizer.texts_to_sequences(quotes)


In [137]:
sequences[0]

[713,
 62,
 29,
 19,
 16,
 946,
 10,
 7,
 5,
 1156,
 8,
 70,
 293,
 10,
 145,
 12,
 809,
 104,
 752,
 70,
 2461]

In [138]:
X=[]
y=[]

In [139]:
for sequence in sequences:
    for i in range(1,len(sequence)):
        input_seq=sequence[:i]
        target=sequence[i]
        X.append(input_seq)
        y.append(target)

In [140]:
len(X)
len(y)


85271

In [141]:
# now do padding to make all input sequences of same length
max_len=max(len(seq) for seq in X)
print("Maximum sequence length:",max_len)

Maximum sequence length: 745


In [142]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded=pad_sequences(X,maxlen=max_len,padding="pre")

In [143]:
X_padded

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1125],
       [   0,    0,    0, ...,   19, 1125,    3],
       [   0,    0,    0, ..., 1125,    3,  169]],
      shape=(85271, 745), dtype=int32)

In [144]:
y=np.array(y)


In [145]:
from tensorflow.keras.utils import to_categorical
y_one_hot=to_categorical(y,num_classes=vocab_size)


In [146]:
y_one_hot.shape

(85271, 8980)

In [147]:
y_one_hot[0]

array([0., 0., 0., ..., 0., 0., 0.], shape=(8980,))

In [148]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM,Dense

In [149]:
embedding_dim=50
rnn_units=128

In [150]:
rnn_model=Sequential()
rnn_model.add(Embedding(input_dim=vocab_size,output_dim=embedding_dim,input_length=max_len))
rnn_model.add(SimpleRNN(rnn_units))
rnn_model.add(Dense(units=vocab_size,activation="softmax"))

d:\Coding\learn\deep_learning_by_sheriyans\venv\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [151]:
rnn_model.compile(optimizer="adam",loss="categorical_crossentropy",metrics=["accuracy"])

In [152]:
rnn_model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [153]:
# now lets build model for lstm


In [154]:
lstm_model=Sequential()
lstm_model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        input_length=max_len
    )
)
lstm_model.add(LSTM(rnn_units))
lstm_model.add(Dense(units=vocab_size,activation="softmax"))

In [155]:
lstm_model.compile(optimizer="adam",loss="categorical_crossentropy",metrics=["accuracy"])

In [156]:
lstm_model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [157]:
epochs=100
batch_size=128


In [158]:
# now all things will be very heavy so i commented the training part

In [159]:
# history_lstm=lstm_model.fit(X_padded,y_one_hot,epochs=epochs,batch_size=batch_size,validation_split=0.1)


In [161]:
# lstm_model.save("lstm_model.h5")

In [ ]:
# i have copied the pretrained model h5 file from the google collab for training

In [163]:
import tensorflow as tf
lstm_model=tf.keras.models.load_model("lstm_model.h5")

In [164]:
index_to_word={}
for word,index in word_index.items():
    index_to_word[index]=word   

In [165]:
index_to_word

{1: 'the',
 2: 'you',
 3: 'to',
 4: 'and',
 5: 'a',
 6: 'i',
 7: 'is',
 8: 'of',
 9: 'that',
 10: 'it',
 11: 'in',
 12: 'be',
 13: 'not',
 14: 'are',
 15: 'your',
 16: 'have',
 17: 'for',
 18: 'but',
 19: 'we',
 20: 'if',
 21: 'what',
 22: 'with',
 23: 'all',
 24: 'love',
 25: 'can',
 26: 'my',
 27: 'when',
 28: 'will',
 29: 'as',
 30: 'who',
 31: 'do',
 32: 'or',
 33: 'me',
 34: 'he',
 35: 'they',
 36: 'life',
 37: 'one',
 38: 'was',
 39: 'like',
 40: 'there',
 41: 'people',
 42: 'on',
 43: 'its',
 44: 'at',
 45: 'so',
 46: 'never',
 47: 'no',
 48: 'them',
 49: 'dont',
 50: 'know',
 51: 'just',
 52: 'more',
 53: 'only',
 54: 'than',
 55: 'because',
 56: 'this',
 57: 'want',
 58: 'up',
 59: 'how',
 60: 'his',
 61: 'things',
 62: 'world',
 63: 'by',
 64: 'think',
 65: 'make',
 66: 'about',
 67: 'time',
 68: 'from',
 69: 'always',
 70: 'our',
 71: 'an',
 72: 'out',
 73: 'us',
 74: 'good',
 75: 'said',
 76: 'she',
 77: 'her',
 78: 'way',
 79: 'go',
 80: 'am',
 81: 'live',
 82: 'has',
 83:

In [166]:
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [173]:
def predictor(model,tokenizer,text,max_len):
    text=text.lower()
    sequence=tokenizer.texts_to_sequences([text])[0]
    sequence=pad_sequences([sequence],maxlen=max_len,padding="pre")

    pred=model.predict(sequence,verbose=0)
    pred_index=np.argmax(pred)
    return index_to_word.get(pred_index,"")



In [175]:
seed_text="what are you"
next_word=predictor(lstm_model,tokenizer,seed_text,max_len)
print(f"Next word after '{seed_text}' is '{next_word}'")

Next word after 'what are you' is 'worrying'


In [176]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
    for _ in range(n_words):
        next_word=predictor(model,tokenizer,seed_text,max_len)
        if(next_word==""):  
            break
        seed_text+= " " + next_word
    return seed_text

In [177]:
seed_text="the meanning of life is"
generated_text=generate_text(lstm_model,tokenizer,seed_text,max_len,n_words=5)
print(generated_text)

the meanning of life is not even a woman is


In [178]:
import pickle
with open("tokenizer.pkl","wb") as f:   
    pickle.dump(tokenizer,f)

In [179]:
with open("max_len.pkl","wb") as f:
    pickle.dump(max_len,f)